# Audio Transcription Setup

Before running the transcription code, we need to set up our environment properly. Here are the required steps:

1. Install required Python packages:
   ```bash
   pip install pydub openai simpleaudio pyaudioop
   ```

2. Install FFmpeg (required for audio processing):
   - Windows: Download from https://ffmpeg.org/download.html and add to PATH
   - Mac: Use `brew install ffmpeg`
   - Linux: Use `sudo apt-get install ffmpeg`

Make sure all these are installed before proceeding with the transcription code.

In [ ]:
# First, let's check if we have required packages
try:
    from openai import OpenAI
    import os
    import math
    print("✓ Basic dependencies loaded successfully")
except ImportError as e:
    print(f"Error loading basic dependencies: {e}")
    
# Try importing audio processing libraries
try:
    from pydub import AudioSegment
    print("✓ pydub loaded successfully")
except ImportError as e:
    print(f"Error loading pydub: {e}")
    print("Please make sure you have FFmpeg installed and all required dependencies")

ModuleNotFoundError: No module named 'pyaudioop'

In [ ]:
# Initialize OpenAI client
import os
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

def transcribe_audio_file(file_path, chunk_duration_minutes=23):
    """
    Transcribe an audio file by splitting it into chunks if necessary
    """
    import subprocess
    import json
    from pathlib import Path
    
    # Get duration using ffprobe
    probe_cmd = [
        'ffprobe', 
        '-v', 'quiet',
        '-print_format', 'json',
        '-show_format',
        file_path
    ]
    
    try:
        # Get file duration using ffprobe
        result = subprocess.run(probe_cmd, capture_output=True, text=True)
        metadata = json.loads(result.stdout)
        duration_seconds = float(metadata['format']['duration'])
        
        # Calculate number of chunks needed
        chunk_duration_seconds = chunk_duration_minutes * 60
        num_chunks = -(-duration_seconds // chunk_duration_seconds)  # Ceiling division
        
        full_transcript = ""
        
        for i in range(int(num_chunks)):
            start_time = i * chunk_duration_seconds
            duration = min(chunk_duration_seconds, duration_seconds - start_time)
            
            # Create temporary file name
            temp_file = f"temp_chunk_{i}.mp3"
            
            # Extract chunk using ffmpeg
            extract_cmd = [
                'ffmpeg',
                '-i', file_path,
                '-ss', str(start_time),
                '-t', str(duration),
                '-acodec', 'libmp3lame',
                '-y',  # Overwrite output files
                temp_file
            ]
            
            try:
                # Extract chunk
                subprocess.run(extract_cmd, capture_output=True, check=True)
                
                # Transcribe chunk
                with open(temp_file, 'rb') as audio_file:
                    transcript = client.audio.transcriptions.create(
                        model="whisper-1",
                        file=audio_file
                    )
                    full_transcript += transcript.text + " "
                    
            finally:
                # Clean up temporary file
                try:
                    Path(temp_file).unlink(missing_ok=True)
                except Exception as e:
                    print(f"Warning: Could not delete temporary file {temp_file}: {e}")
        
        return full_transcript.strip()
        
    except subprocess.CalledProcessError as e:
        print(f"Error running ffmpeg/ffprobe: {e}")
        raise
    except Exception as e:
        print(f"Error during transcription: {e}")
        raise

# Use the function
try:
    audio_path = "WhatsApp Audio 2025-11-03 at 7.45.17 PM.mp4"
    transcript = transcribe_audio_file(audio_path)
    print("Full Transcript:")
    print(transcript)
except Exception as e:
    print(f"Error: {e}")